# 19 — Topic info per run

Displays `topic_id`, top words, review count, aspect label, sentiment, and confidence
for every run — coast bands and year slices.

Useful for inspecting why topics were labeled `other`, checking labeling quality,
and understanding what BERTopic found in each segment.

In [ ]:
import sys
from pathlib import Path
sys.path.append("../src")

import duckdb
import pandas as pd
from IPython.display import display, Markdown
import llm_label as ll

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

In [ ]:
con = duckdb.connect(str(ll.DB_PATH), read_only=True)

df = con.execute("""
    SELECT
        tl.run_id,
        tl.topic_id,
        COUNT(rt.review_id)  AS n_reviews,
        tl.top_words,
        ta.key_aspect,
        ta.sentiment,
        ROUND(ta.weight, 2)  AS weight,
        ta.sub_aspects,
        ROUND(tl.confidence, 2) AS confidence,
        tl.short_reason
    FROM TOPIC_LABELS tl
    LEFT JOIN REVIEW_TOPICS rt
           ON rt.run_id = tl.run_id AND rt.topic_id = tl.topic_id
    LEFT JOIN TOPIC_ASPECTS ta
           ON ta.run_id = tl.run_id AND ta.topic_id = tl.topic_id
           AND ta.weight = (
               SELECT MAX(weight) FROM TOPIC_ASPECTS
               WHERE run_id = tl.run_id AND topic_id = tl.topic_id
           )
    WHERE tl.topic_id != -1
    GROUP BY tl.run_id, tl.topic_id, tl.top_words,
             ta.key_aspect, ta.sentiment, ta.weight,
             ta.sub_aspects, tl.confidence, tl.short_reason
    ORDER BY tl.run_id, tl.topic_id
""").df()

con.close()

# Keep top 15 representation words, drop empty trailing slots
def fmt_words(s):
    words = [w.strip() for w in str(s).split(",") if w.strip()]
    return " · ".join(words[:15])

df["representation"] = df["top_words"].apply(fmt_words)
df = df.drop(columns="top_words")

run_ids = sorted(df["run_id"].unique())
print(f"{len(df):,} topics across {len(run_ids)} runs")
print("Runs:", run_ids)

## Coast band runs

In [ ]:
import json

ASPECT_COLORS = {
    "location":    "#4878a8",
    "room":        "#6a9f58",
    "cleanliness": "#5bb5c8",
    "service":     "#e49444",
    "food_drink":  "#d1605e",
    "value":       "#a87ca8",
    "facilities":  "#8c7b6b",
    "other":       "#aaaaaa",
}


def color_aspect(val):
    color = ASPECT_COLORS.get(val, "#aaaaaa")
    return f"background-color: {color}22; color: {color}; font-weight: bold"


def fmt_sub_aspects(val):
    if pd.isna(val) or val is None:
        return "—"
    try:
        items = json.loads(val)
        return ", ".join(items) if items else "—"
    except Exception:
        return str(val)


def show_run(run_id: str) -> None:
    d = (df[df["run_id"] == run_id]
         .drop(columns="run_id")
         .reset_index(drop=True))
    d["sub_aspects"] = d["sub_aspects"].apply(fmt_sub_aspects)

    cols = ["topic_id", "n_reviews", "key_aspect", "sentiment",
            "weight", "sub_aspects", "confidence", "short_reason", "representation"]
    d = d[[c for c in cols if c in d.columns]]

    n_other = (d["key_aspect"] == "other").sum()
    display(Markdown(
        f"### `{run_id}` — {len(d)} topics · "
        f"{d['n_reviews'].sum():,} reviews "
        f"({n_other} labeled *other*)"
    ))

    styled = (
        d.style
        .map(color_aspect, subset=["key_aspect"])
        .format({"n_reviews": "{:,}", "weight": "{:.2f}", "confidence": "{:.2f}"})
        .set_properties(**{"font-size": "11px", "white-space": "pre-wrap"})
        .set_properties(subset=["representation"], **{"min-width": "420px", "color": "#555"})
        .set_table_styles([{"selector": "th", "props": [("font-size", "11px")]}])
    )
    display(styled)


coast_runs = [r for r in run_ids if r.startswith("coast_band_")]
for run_id in coast_runs:
    show_run(run_id)

## Year runs

In [ ]:
year_runs = [r for r in run_ids if r.startswith("year_")]
for run_id in year_runs:
    show_run(run_id)

## Summary — `other` topics across all runs

Topics labeled `other` are sentiment-only or too noisy to map to any specific aspect.
High counts here indicate BERTopic found weak clusters for that segment.

In [ ]:
summary = (
    df.groupby("run_id")
    .apply(lambda d: pd.Series({
        "total_topics":   len(d),
        "other_topics":   (d["key_aspect"] == "other").sum(),
        "total_reviews":  d["n_reviews"].sum(),
        "other_reviews":  d.loc[d["key_aspect"] == "other", "n_reviews"].sum(),
        "other_pct":      round(100 * (d["key_aspect"] == "other").sum() / len(d), 1),
    }))
    .reset_index()
    .sort_values("other_pct", ascending=False)
)

summary.style \
    .format({"total_reviews": "{:,.0f}", "other_reviews": "{:,.0f}",
             "other_pct": "{:.1f}%"}) \
    .background_gradient(subset=["other_pct"], cmap="YlOrRd") \
    .set_caption("Runs ranked by % of topics labeled 'other'")